# 2.2 Generate the Kilosort channel map of a processed session

**[env: optonpx]**

Writes one Kilosort 4 channel map (`.mat`) per probe for the concatenated recording made by
notebook 2.1, next to its `continuous.dat`:
`<processed_session>/<probe>/<session_name>_<ProbeX>.mat`.

- **Session**: set `session_name` (the metadata file `metadata/<session_name>.yaml`) and the same
  `base_dir_*` / `processed_suffix` as in notebook 2.1, so that the processed folder matches.
- **Bad channels** come only from the metadata (`<probe>.kilosort_params.bad_channels`, 0-indexed,
  Open Ephys numbering). To change them, edit the metadata and re-run this notebook; notebook 2.1
  does not need to be re-run.
- **Geometry** (x/y position, shank of every channel) is read from the raw recordings'
  `settings.xml` (read-only). Every session that was concatenated must have the same wiring.
- **Safety**: the processed folder must already exist (run notebook 2.1 first; 2.1 refuses to
  write into an existing folder). If Kilosort results already exist for a probe, a new channel map
  would no longer match them, so the notebook stops unless `is_overwrite = True`. Before an
  existing `.mat` is replaced, the change in excluded channels is printed.

In [ ]:
from pathlib import Path

# ========== Specify the session ==========
session_name = "IM-1979_2026-09-03"
base_dir_raw = Path(r"D:/")
base_dir_output = Path(r"E:/")
processed_suffix = ""   # must match notebook 2.1
is_overwrite = False              # True: write even if Kilosort results already exist for a probe
# =========================================

In [ ]:
import sys

import numpy as np
import scipy.io
import yaml

sys.path.append("..")
from src import oe_parse_folders, oe_parse_params, save_kilosort_channel_map

metadata_path = Path.cwd().parent / "metadata" / f"{session_name}.yaml"
print(metadata_path)
with open(metadata_path) as f:
    config = yaml.safe_load(f)

animal_dir = base_dir_raw / config["animal"]
processed_session = base_dir_output / config["animal"] / "ephys" / (config["sessions"]["processed"] + processed_suffix)
probe_names = config["streams"]["probes"]
print(f"processed session: {processed_session}")
print(f"probes: {', '.join(probe_names)}")

if not processed_session.exists():
    raise FileNotFoundError(f"{processed_session} does not exist -- run notebook 2.1 first "
                            "(it refuses to write into an existing folder)")

In [ ]:
# --- per probe: geometry from settings.xml of every concatenated session (read-only), must be identical ---
def match_one_stream(items, stream_name, name_of):
    """Return the single item whose name ends with `stream_name`, case-insensitively
    (the metadata says 'probeA', Open Ephys says 'ProbeA')."""
    matches = [item for item in items
               if str(name_of(item)).split(".")[-1].lower() == stream_name.lower()]
    if len(matches) != 1:
        raise ValueError(f"probe {stream_name!r}: expected 1 match, got {[name_of(i) for i in items]}")
    return matches[0]


session_params = {}   # role -> list of probe params parsed from that session's settings.xml
for role in {role for probe_name in probe_names for role in config[probe_name]["preprocess"]["concatenate"]}:
    _, oe_paths = oe_parse_folders(animal_dir / "ephys" / config["sessions"][role])
    session_params[role], _ = oe_parse_params(oe_paths["xml"], oe_paths["oebin"])

probe_params = {}     # probe_name -> params of the first concatenated session
for probe_name in probe_names:
    order = config[probe_name]["preprocess"]["concatenate"]
    params = [match_one_stream(session_params[role], probe_name, lambda p: p["stream_name"]) for role in order]
    for role, other in zip(order[1:], params[1:]):
        for field in ("channel_count", "channel_shank", "channel_xpos", "channel_ypos", "channel_electrode"):
            if other[field] != params[0][field]:
                raise ValueError(f"{probe_name}: {field} differs between {order[0]} and {role} "
                                 "-- different probe configuration, the channel map would be wrong")
    probe_params[probe_name] = params[0]
    print(f"{probe_name}: {params[0]['stream_name']}, {params[0]['channel_count']} channels, "
          f"geometry identical in {', '.join(order)}")

In [ ]:
# --- checks before writing: bad channels, existing channel map, existing Kilosort results ---
blocked = []
for probe_name in probe_names:
    params = probe_params[probe_name]
    n_chan = int(params["channel_count"])
    bad_channels = sorted(config[probe_name]["kilosort_params"]["bad_channels"])
    out_dir = processed_session / probe_name
    mat_path = out_dir / f"{session_name}_{params['stream_name']}.mat"

    out_of_range = [ch for ch in bad_channels if not 0 <= ch < n_chan]
    if out_of_range:
        raise ValueError(f"{probe_name}: bad_channels outside 0..{n_chan - 1}: {out_of_range}")
    if not (out_dir / "continuous.dat").exists():
        raise FileNotFoundError(f"{out_dir / 'continuous.dat'} not found -- run notebook 2.1 first")

    shanks = np.array([params["channel_shank"][ch] for ch in range(n_chan) if ch not in set(bad_channels)])
    per_shank = ", ".join(f"shank {s}: {int((shanks == s).sum())}" for s in np.unique(shanks))
    print(f"{probe_name}: {n_chan - len(bad_channels)} of {n_chan} channels kept ({per_shank})")
    print(f"  bad_channels (metadata): {bad_channels}")

    if mat_path.exists():
        kept_before = set((scipy.io.loadmat(mat_path)["chanMap"].ravel() - 1).astype(int).tolist())
        excluded_before = sorted(set(range(n_chan)) - kept_before)
        if excluded_before == bad_channels:
            print(f"  existing {mat_path.name}: same excluded channels")
        else:
            print(f"  existing {mat_path.name}: excluded {excluded_before}\n"
                  f"    newly excluded: {sorted(set(bad_channels) - set(excluded_before))}, "
                  f"no longer excluded: {sorted(set(excluded_before) - set(bad_channels))}")

    kilosort_dir = out_dir / config[probe_name]["kilosort_params"].get("output_dir", "kilosort4")
    if kilosort_dir.exists() and any(kilosort_dir.iterdir()):
        print(f"  Kilosort results exist: {kilosort_dir}")
        blocked.append(probe_name)

if blocked and not is_overwrite:
    raise RuntimeError(f"Kilosort results already exist for {blocked}: a new channel map would not match them. "
                       "Set is_overwrite = True to write anyway (and re-run Kilosort).")

In [ ]:
# --- write one .mat per probe ---
for probe_name in probe_names:
    mat_path = save_kilosort_channel_map(
        probe_params[probe_name], processed_session / probe_name, session_name,
        bad_channels=config[probe_name]["kilosort_params"]["bad_channels"], overwrite=True)
    kept = scipy.io.loadmat(mat_path)["chanMap"].ravel().astype(int) - 1     # read back, 0-indexed
    assert set(kept).isdisjoint(config[probe_name]["kilosort_params"]["bad_channels"])
    print(f"  {mat_path}")